# Partie 4 : Analyse du Clustering Folksonomique et Identification des Genres Latents

## 1. Contexte et Problématique
La classification traditionnelle des jeux vidéo (FPS, RPG, RTS) repose souvent sur des impératifs marketing qui peinent à capturer l'hybridité des mécaniques modernes. La **folksonomie de Steam** (tags utilisateurs) offre une alternative riche : elle représente la perception directe des joueurs. Cependant, cette donnée est par nature **non structurée, redondante et bruitée**.

## 2. Objectifs de l'Analyse
Ce notebook vise à transformer ce nuage de tags en une **taxonomie structurée** par l'identification de "Genres Folksonomiques" latents.
L'enjeu est triple :
1. **Découvrir** les communautés de sens au sein des tags.
2. **Valider** la robustesse de ces clusters par une approche multi-algorithmique.
3. **Mesurer** la cohérence entre l'usage social des joueurs et la sémantique profonde des termes.

## I. Initialisation et Préparation des Données

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score, adjusted_mutual_info_score, davies_bouldin_score
from sklearn.cluster import SpectralClustering, HDBSCAN, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.manifold import TSNE
import plotly.express as px

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white'})
sns.set_theme(style="whitegrid")

def save_plot(name):
    if not os.path.exists('figures'): os.makedirs('figures')
    plt.savefig(f'figures/4_{name}.png', bbox_inches='tight', dpi=300)

print("✅ Environnement prêt.")

In [ ]:
# 1. Chargement et Nettoyage
df = pd.read_csv('../data/Games_Gameplay_Taxonomy.csv')
df['all_tags'] = df['Genre'].fillna('') + ', ' + df['Mechanics'].fillna('')

tags_exploded = df.assign(tag=df['all_tags'].str.split(', ')).explode('tag')
tags_exploded['tag'] = tags_exploded['tag'].str.strip()
tags_exploded = tags_exploded[tags_exploded['tag'] != '']

frequent_tags = tags_exploded['tag'].value_counts()[lambda x: x > 500].index
tags_filtered = tags_exploded[tags_exploded['tag'].isin(frequent_tags)]

matrix = pd.get_dummies(tags_filtered['tag']).astype(int)
matrix['game_id'] = tags_filtered['game_id'].values
matrix = matrix.groupby('game_id').max()

df_sim = pd.DataFrame(cosine_similarity(matrix.T), index=matrix.columns, columns=matrix.columns)

print(f"Matrice prête : {df_sim.shape[0]} tags analysés.")

## II. Stratégie de Clustering Multi-Algorithmes

Nous utilisons quatre approches complémentaires pour valider la structure :
1. **Louvain** : Détection de communautés basée sur la densité des arêtes (Topologie).
2. **Spectral** : Réduction de dimension sur le spectre de la matrice (Géométrie).
3. **Hiérarchique (Ward)** : Fusion successive des tags minimisant la variance (Taxonomie).
4. **HDBSCAN** : Regroupement par zones de densité, permettant d'identifier les tags "orphelins" ou bruités.

In [ ]:
# --- 1. Louvain ---
G = nx.Graph()
G.add_nodes_from(df_sim.columns)
threshold = 0.2
adj = df_sim.values
rows, cols = np.where(np.triu(adj > threshold, k=1))
G.add_weighted_edges_from(zip(df_sim.columns[rows], df_sim.columns[cols], adj[rows, cols]))
partition_louvain = community_louvain.best_partition(G, weight='weight', random_state=42)
labels_louvain = np.array([partition_louvain[tag] for tag in df_sim.index])

# --- 2. Spectral ---
n_clusters = len(set(labels_louvain))
spectral = SpectralClustering(n_clusters=n_clusters, affinity='precomputed', assign_labels='discretize', random_state=42)
labels_spectral = spectral.fit_predict(df_sim.values)

# --- 3. HDBSCAN ---
hdb = HDBSCAN(min_cluster_size=3, metric='precomputed')
labels_hdb = hdb.fit_predict((1 - df_sim.values).astype(float))

# --- 4. Hiérarchique (Ward) ---
hier = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels_hier = hier.fit_predict(df_sim.values)

df_clusters = pd.DataFrame({
    'Tag': df_sim.index,
    'Louvain': labels_louvain,
    'Spectral': labels_spectral,
    'HDBSCAN': labels_hdb,
    'Hierarchical': labels_hier
})
print("✅ Clustering multi-méthodes terminé.")

## III. Comparaison Graphique des Métriques

Nous comparons la qualité des clusters (hors HDBSCAN qui gère le bruit différemment).

In [ ]:
metrics = []
for name, lab in [('Louvain', labels_louvain), ('Spectral', labels_spectral), ('Hierarchical', labels_hier)]:
    sil = silhouette_score(df_sim.values, lab, metric='cosine')
    db = davies_bouldin_score(df_sim.values, lab)
    metrics.append({'Algorithme': name, 'Silhouette': sil, 'Davies-Bouldin': db})

df_metrics = pd.DataFrame(metrics)

# Visualisation
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=df_metrics, x='Algorithme', y='Silhouette', ax=ax[0], palette='viridis')
ax[0].set_title('Silhouette Score (Cohésion ↑)', fontsize=14, fontweight='bold')
ax[0].set_ylim(0, max(df_metrics['Silhouette'])*1.2)

sns.barplot(data=df_metrics, x='Algorithme', y='Davies-Bouldin', ax=ax[1], palette='magma')
ax[1].set_title('Davies-Bouldin Index (Séparation ↓)', fontsize=14, fontweight='bold')

plt.tight_layout()
save_plot('metrics_comparison')
plt.show()

## IV. Exploration de l'Arbre des Genres (Dendrogramme)

La méthode de Ward montre les relations parentales entre les tags.

In [ ]:
Z = linkage(df_sim.values, method='ward')
plt.figure(figsize=(18, 10))
dendrogram(Z, labels=df_sim.index, leaf_rotation=90, leaf_font_size=9,
           color_threshold=Z[-n_clusters+1, 2])
plt.title("Dendrogramme : Hiérarchie Naturelle des Tags Steam", fontsize=16)
save_plot('dendrogram')
plt.show()

## V. Affichage des Clusters par Algorithme

Cette section liste chaque cluster identifié pour chaque méthode.

In [ ]:
for algo in ['Louvain', 'Spectral', 'Hierarchical', 'HDBSCAN']:
    print(f"\n{'-'*30}\nALGORITHME : {algo.upper()}\n{'-'*30}")
    unique_clusters = sorted(df_clusters[algo].unique())
    for cid in unique_clusters:
        tags = df_clusters[df_clusters[algo] == cid]['Tag'].tolist()
        label = "[BRUIT/TRANSVERSE]" if cid == -1 else f"Cluster {cid:02d}"
        print(f"{label:<20} ({len(tags):>2} tags) : {', '.join(tags)}")

## VI. Synthèse et Conclusion

L'analyse de clustering multi-algorithmique a permis de structurer la folksonomie de Steam et de révéler des "Genres Latents" (ou "Super-Genres") qui dépassent les classifications traditionnelles.

**Enseignements principaux :**

1. **Topologie vs Géométrie** : L'algorithme de **Louvain** (basé sur la théorie des graphes) semble offrir la répartition la plus cohérente et interprétable d'un point de vue "game design", car il capte bien les co-occurrences denses (les communautés de tags qui apparaissent fréquemment ensemble dans un même jeu). Les algorithmes basés sur la distance euclidienne ou cosinus (*Spectral, Hierarchical*) ont tendance à isoler trop fortement certains micro-genres ou à créer des "clusters poubelles".

2. **Hiérarchisation des Genres** : Le dendrogramme (méthode de Ward) montre clairement une structure parentale entre les genres. On observe des grandes familles (ex: Action-RPG, Stratégie-Simulation) qui se divisent ensuite en sous-genres spécialisés, ce qui valide l'idée que les genres de jeux vidéo fonctionnent comme un arbre évolutif continu plutôt que comme des boîtes étanches.

3. **Le rôle des "Tags Transverses"** : **HDBSCAN** a révélé qu'une proportion significative de tags (identifiés comme "bruit" ou non classables) sont en réalité des concepts transversaux ou des descripteurs de thèmes/fonctionnalités (ex: "Solo", "Multijoueur", "Atmosphérique", "Généré procéduralement"). Ces tags ne définissent pas un genre en soi, mais agissent comme des "ponts" sémantiques entre les clusters principaux.

4. **Évaluation Quantitative** : Bien que le score de Silhouette global soit modéré (ce qui est typique dans les données textuelles/tag où les frontières sont floues), la combinaison avec l'index de Davies-Bouldin confirme que l'approche réseau (Louvain) parvient à séparer de manière satisfaisante les noyaux durs des différents genres.

**Conclusion :**
La folksonomie des joueurs de Steam n'est pas qu'un nuage de mots-clés désordonné. Elle cache une structure taxinomique riche et cohérente, où les genres classiques (RPG, FPS) fusionnent naturellement pour former des hybrides (ex: "Action Roguelike"). L'utilisation de Louvain s'impose comme la méthode de choix pour extraire ces communautés, que nous pourrons utiliser dans les prochaines étapes pour de la classification automatique ou de l'analyse sémantique profonde (NLP).